# Imports

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *
from src.viterbi import viterbi

import math
import random
from scipy.stats import ttest_ind
from pprint import pprint

# Train HMM

In [2]:
states = ["A", "N"]
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM(states)
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename)

# Small test sample

In [3]:
test_file = "../data/test.fa"
gene_dict = load_fasta_dict(test_file)
results = analyze_genes(gene_dict, hmm)
pprint(results)

{'Gene_1': (0.85, 'AAAAAAAAAAAAAAAAANNN'),
 'Gene_2': (0.75, 'NNNNNAAAAAAAAAAAAAAA'),
 'Gene_3': (0.25, 'AAAAANNNNNNNNNNNNNNN'),
 'Gene_4': (0.4, 'NNNNNNNNNNNNAAAAAAAA')}


# Synthetic Data

In [4]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    

codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["A"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 21):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)


# Test with synthetic data

In [5]:
results_human = analyze_genes(human_like_seq, hmm)
results_random = analyze_genes(random_seq, hmm)

human_scores = []
random_scores = []
for score, _ in results_human.values():
    human_scores.append(score)

for score, _ in results_random.values():
    random_scores.append(score)

human_avg = sum(human_scores)/len(human_scores)
random_avg = sum(random_scores)/len(random_scores)

print("Human sequences:")
pprint(results_human, sort_dicts=False)
print("Random sequences:")
pprint(results_random, sort_dicts=False)

print(f"Human-like average: {human_avg}")
print(f"Random average: {random_avg}")

Human sequences:
{'h1': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h2': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h3': (0.65, 'NNNNNNNAAAAAAAAAAAAA'),
 'h4': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h5': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h6': (0.65, 'NNNNNNNAAAAAAAAAAAAA'),
 'h7': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h8': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h9': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h10': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h11': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h12': (0.45, 'NNNNNAAAAAAAAANNNNNN'),
 'h13': (0.75, 'NNNNNAAAAAAAAAAAAAAA'),
 'h14': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h15': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h16': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h17': (0.65, 'AAAAAAAAAAAAANNNNNNN'),
 'h18': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h19': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h20': (0.25, 'NNNNNNNNNNNNNNNAAAAA')}
Random sequences:
{'r1': (0.25, 'NNNNNNNNNNNNNNNAAAAA'),
 'r2': (0.0, 'NNNNNNNNNNNNNNNNNNNN'),
 'r3': (0.0, 'NNNNNNNNNNNNNNNNNNNN'),
 'r4': (0.0, 'NNNNNNNNNNNNNNNNNNNN'),
 'r5': (0.55, 'NNNNNNAAAAAAAAAAANNN

# Statistical test

In [6]:
t_stat, p_val = ttest_ind(human_scores, random_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

t-stat: 7.333304864877999
p-value: 8.74574709167596e-09
